# 📚 Agente Estudos — gerar material num notebook (Kaggle / Colab)

Roda o pipeline do `agente_estudos` com um **LLM maior do que cabe no seu PC**, e devolve a pasta de estudos zipada para você colocar no vault Obsidian.

**Antes de rodar**
- **Kaggle:** *Settings → Accelerator → GPU T4 x2* e *Internet → On*. Segredos em *Add-ons → Secrets*.
- **Colab:** *Ambiente de execução → Alterar tipo → GPU T4*. Segredos no ícone 🔑 da barra lateral (dar acesso ao notebook).
- O notebook clona o repositório do GitHub, então **o código precisa estar com push feito**. Se o repo for privado, crie o segredo `GITHUB_TOKEN`.

**Três caminhos de LLM** (célula 1, `LLM_MODO`):
| Modo | O que é | Precisa de |
|---|---|---|
| `ollama_local` | Sobe o Ollama na GPU do notebook com um modelo maior (ex. `qwen2.5:14b`) | GPU T4 (nada pago) |
| `groq` | API do Groq (camada gratuita, modelos 70B) — já suportada pelo projeto | segredo `GROQ_API_KEY` |
| `openai_compat` | Qualquer endpoint compatível com a API OpenAI (ex. Gemini, OpenRouter, vLLM) | `OPENAI_COMPAT_BASE_URL` + segredo `OPENAI_COMPAT_API_KEY` |

> Ainda é preciso testar qual modelo dá o melhor material **em pt-BR** — compare 2-3 rodadas com o mesmo tema antes de fixar um.

In [ ]:
# ── 1. Configuração (edite aqui) ──────────────────────────────────────────────
TEMA     = "Devops"
FOCO     = "Fundamentos de DevOps e práticas essenciais"
DIDATICA = ""                      # vazio = didática padrão do projeto
FONTES   = [                       # URLs (YouTube, artigos, papers, GitHub) — pode ficar vazio
    # "https://www.youtube.com/watch?v=...",
]
MODO_PIPELINE = "pipeline"         # "pipeline" (sem tool calling — mais robusto) | "agent"

LLM_MODO = "ollama_local"          # "ollama_local" | "groq" | "openai_compat"
OLLAMA_MODELO   = "qwen2.5:14b"    # cabe numa T4 16GB quantizado; teste também qwen3:14b, gemma3:12b
OLLAMA_CONTEXTO = 16384            # janela de contexto (o padrão do Ollama é pequeno e trunca os prompts)
GROQ_MODELO     = "llama-3.3-70b-versatile"
OPENAI_COMPAT_BASE_URL = ""        # ex.: "https://generativelanguage.googleapis.com/v1beta/openai/"
OPENAI_COMPAT_MODELO   = ""
TEMPERATURA = 0.7

REPO_URL = "https://github.com/Roberto207/agente_estudos"
BRANCH   = "main"

In [ ]:
# ── 2. Ambiente, segredos e pastas ────────────────────────────────────────────
import os, re, subprocess, sys, time, unicodedata
from pathlib import Path

NO_KAGGLE = Path("/kaggle/working").exists()
NO_COLAB = "google.colab" in sys.modules
WORK = Path("/kaggle/working") if NO_KAGGLE else Path("/content") if NO_COLAB else Path.cwd()
print("Ambiente:", "Kaggle" if NO_KAGGLE else "Colab" if NO_COLAB else "local", "| trabalho em", WORK)

def segredo(nome: str) -> str:
    """Lê um segredo do Kaggle Secrets / Colab userdata / variável de ambiente."""
    if NO_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(nome)
        except Exception:
            pass
    if NO_COLAB:
        try:
            from google.colab import userdata
            return userdata.get(nome)
        except Exception:
            pass
    return os.environ.get(nome, "")

def slugify(s: str) -> str:
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]+", "_", s.lower()).strip("_") or "estudo"

PASTA = WORK / "estudos" / slugify(TEMA)
print("Pasta de saída:", PASTA)

In [ ]:
# ── 3. Código do projeto + dependências ───────────────────────────────────────
REPO_DIR = WORK / "agente_estudos"
if not REPO_DIR.exists():
    token = segredo("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
    r = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, str(REPO_DIR)],
                       capture_output=True, text=True)
    if r.returncode != 0:  # não imprime a URL (pode conter o token)
        raise RuntimeError("git clone falhou — repo privado sem GITHUB_TOKEN? " + r.stderr.replace(token or "\0", "***")[-500:])
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)

# Só o núcleo: a interface web (fastapi/uvicorn/jinja2) não é usada aqui.
reqs = [l.strip() for l in (REPO_DIR / "requirements.txt").read_text().splitlines()
        if l.strip() and not l.startswith("#")
        and not re.match(r"(fastapi|uvicorn|jinja2|python-multipart)", l.strip())]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *reqs], check=True)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("ok —", len(reqs), "dependências")

In [ ]:
# ── 4. LLM: sobe o Ollama na GPU (só no modo ollama_local) ────────────────────
if LLM_MODO == "ollama_local":
    if subprocess.run(["which", "ollama"], capture_output=True).returncode != 0:
        subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
    env = {**os.environ, "OLLAMA_CONTEXT_LENGTH": str(OLLAMA_CONTEXTO), "OLLAMA_HOST": "127.0.0.1:11434"}
    ollama_proc = subprocess.Popen(["ollama", "serve"], env=env,
                                   stdout=open(WORK / "ollama.log", "w"), stderr=subprocess.STDOUT)
    import requests
    for _ in range(60):
        try:
            requests.get("http://127.0.0.1:11434/api/tags", timeout=2); break
        except Exception:
            time.sleep(1)
    else:
        raise RuntimeError("Ollama não subiu — veja ollama.log")
    print(f"Baixando {OLLAMA_MODELO} (alguns minutos na primeira vez)...")
    subprocess.run(["ollama", "pull", OLLAMA_MODELO], check=True, env=env)
    print("Ollama pronto.")

In [ ]:
# ── 5. Monta o config (equivalente ao config.yaml, sem importar main.py) ──────
config = {
    "search": {"max_results": 5},
    "transcription": {"provider": "auto", "groq_api_key": segredo("GROQ_API_KEY"), "local_model": "base"},
    "json_retries": 2,
    "content": {"min_words": 900, "rewrite_attempts": 1},
}
if LLM_MODO == "ollama_local":
    config["provider"] = "ollama"
    config["ollama"] = {"base_url": "http://127.0.0.1:11434", "model": OLLAMA_MODELO, "temperature": TEMPERATURA}
elif LLM_MODO == "groq":
    config["provider"] = "groq"
    config["groq"] = {"api_key": segredo("GROQ_API_KEY"), "model": GROQ_MODELO, "temperature": TEMPERATURA}
elif LLM_MODO == "openai_compat":
    config["provider"] = "openai"
    config["openai"] = {"api_key": segredo("OPENAI_COMPAT_API_KEY"), "base_url": OPENAI_COMPAT_BASE_URL,
                        "model": OPENAI_COMPAT_MODELO, "temperature": TEMPERATURA}
else:
    raise ValueError(f"LLM_MODO desconhecido: {LLM_MODO}")

prov = config["provider"]
assert prov == "ollama" or config[prov].get("api_key"), f"Segredo da API ausente para o modo {LLM_MODO}"
print("Provider:", prov, "| modelo:", config[prov]["model"])

In [ ]:
# ── 6. Teste rápido do LLM (falha cedo, antes de 20 min de pipeline) ──────────
from src.llm import LLMClient
t0 = time.time()
resp = LLMClient(config).chat([{"role": "user", "content": "Responda só: ok"}], max_tokens=10)
print(repr(resp), f"— {time.time() - t0:.1f}s")

In [ ]:
# ── 7. Gera o material ────────────────────────────────────────────────────────
PASTA.mkdir(parents=True, exist_ok=True)
if MODO_PIPELINE == "agent":
    from src.agent import run
else:
    from src.pipeline import run
t0 = time.time()
run(config=config, tema=TEMA, foco=FOCO, didatica=DIDATICA, pasta=str(PASTA), fontes=FONTES)
print(f"\nConcluído em {(time.time() - t0) / 60:.1f} min")

In [ ]:
# ── 8. Confere o tamanho dos arquivos e zipa para download ────────────────────
import shutil
mds = sorted(p for p in PASTA.rglob("*.md") if "transcripts" not in p.parts)
for p in mds:
    print(f"{len(p.read_text(encoding='utf-8').split()):>6} palavras  {p.relative_to(PASTA)}")
zip_path = shutil.make_archive(str(WORK / slugify(TEMA)), "zip", PASTA)
print("\nZip:", zip_path)
if NO_COLAB:
    from google.colab import files
    files.download(zip_path)
elif NO_KAGGLE:
    print("Baixe pela aba Output do notebook (lado direito) → arquivo .zip")

### Depois de baixar
1. Descompacte o zip dentro do vault (ex. `~/obsidian/<area>/<tema>`).
2. Opcional: rode localmente `python main.py --mode postprocess --pasta <pasta>` se quiser regerar flashcards/quiz com outro modelo.
3. O índice RAG (`.rag_index.json`) é criado localmente na primeira vez que você abrir o chat sobre a pasta.